# cancer-recon-apoptosis — Step 3: Anchor specificity / safety audit (Colab)

**Question (falsifiable):** does any candidate ANCHOR receptor have a real therapeutic window — tissue-restricted AND low in the *essential parenchyma of vital organs* — so a DR5-clustering binder spares healthy tissue?

**Method (critique-hardened — see `docs/methodology/STEP3_METHODOLOGY.md`):** two-axis metric = GTEx τ (tissue-specificity) + HPA cell-type IHC in vital parenchyma. Thresholds **calibrated on a labelled benchmark** of validated targets; the run is trusted only if it reproduces ground truth (HER2→FAIL on heart; good benchmark separates from bad; DR5 non-selective).

**Runtime:** CPU, no GPU. Downloads ~14 MB (GTEx + HPA). Run all → `data/specificity/specificity_audit.csv` + a commit-stamped run log.

In [ ]:
#@title Cell 1 — clone / pull repo
print('[CELL 1] ▶ clone or pull repo')
import os
from pathlib import Path
REPO_URL = 'https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git'
REPO_DIR = Path('/content/cancer-recon-apoptosis')
if REPO_DIR.exists():
    !cd {REPO_DIR} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git log --oneline -1
assert (REPO_DIR / 'scripts' / '07_specificity_audit.py').exists()
print('[CELL 1] ✓ done')

In [ ]:
#@title Cell 2 — start run log + install deps
import sys
from scripts.runlog import new_log, run_logged, finalize
RUNLOG = new_log('step3', repo_dir='.')
import importlib.util
if importlib.util.find_spec('pandas') is None:
    run_logged([sys.executable,'-m','pip','install','-q','pandas','numpy'], RUNLOG, label='pip install pandas numpy')
else:
    print('  pandas/numpy present')
print('[CELL 2] ✓ done')

In [ ]:
#@title Cell 3 — run the specificity audit (downloads GTEx + HPA ~14 MB)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = globals().get('RUNLOG') or new_log('step3', repo_dir='.')
rc = run_logged([sys.executable, '-u', 'scripts/07_specificity_audit.py'], RUNLOG)
print(f'[CELL 3] exit={rc}', '✓' if rc==0 else '✗ (see RUN-TRUST controls in log)')
import os, pandas as pd
p = 'data/specificity/specificity_audit.csv'
if os.path.exists(p):
    print('\nFull audit table:')
    print(pd.read_csv(p).to_string(index=False))

In [ ]:
#@title Cell 4 — finalize + download run log
from scripts.runlog import new_log, finalize
RUNLOG = globals().get('RUNLOG') or new_log('step3', repo_dir='.')
finalize(RUNLOG)
!ls -la runs/logs data/specificity/*.csv 2>/dev/null

## Interpreting the result

- **Run-trust controls must all PASS** (HER2→FAIL, good benchmark PASS≥5/7, bad benchmark FAIL=5/5, DR5 non-selective) or verdicts are withheld.
- **PASS** = clean window (tissue-restricted, low vital-parenchyma protein). **FAIL** = no window. **CAUTION** = marginal.
- Expected (real data, 2026-05-29): all 10 Step-2 candidates FAIL; the validated benchmark targets PASS — so the test discriminates and the candidate failure is a true finding. Next: pivot the anchor to a tissue-restricted antigen (Trop2/NECTIN4 for breast, MSLN/FOLR1/DLL3 for lung, CLDN18.2 for GI) and/or combinatorial logic-gating, then Step 4 reward.